# 23 Local Bluesky Text Preparation

This notebook performs deterministic local-only Bluesky post-text preparation for Phase 23.

- No Snowflake access
- No reruns of firehose/hydration/enrichment
- Raw source artifacts remain unchanged


**Notebook purpose:** Discovers the best local Bluesky run root, extracts post text from raw/hydrated JSONL files, and produces a deterministic prepared-posts parquet with text features (clean text, hashtag/URL/mention flags, token counts).

**Required data:** Bluesky pipeline capture + hydration output files (raw_posts and hydrated_posts JSONL.gz directories under `data/`). Requires the firehose and hydration pipelines to have been run at least once.

**Run order:** Run after notebook 22 (trend normalization). Run before notebook 24 (topic extraction).

## 1. Discover and Select Local Source

We scan local run roots, evaluate usable post-text coverage, and select the best source deterministically.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'src' / 'nlp' / 'post_normalization.py').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing src/nlp/post_normalization.py')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.nlp.post_normalization import (
    select_best_local_source_candidate,
    prepare_from_best_local_source,
    summarize_prepared_posts,
)

bluesky_source_root = ROOT / "data"

# Guard: pipeline data must exist
if not bluesky_source_root.exists() or not any(bluesky_source_root.iterdir()):
    print("DATA NOT YET AVAILABLE -- run the firehose capture pipeline first. Skipping this cell.")
else:
    selection = select_best_local_source_candidate(base_dir=bluesky_source_root)
    selected = selection["selected"]

    print("Selected run root:", selected["run_root"])
    print("Raw files:", len(selected["raw_paths"]))
    print("Hydrated files:", len(selected["hydrated_paths"]))
    print("Prepared non-empty text count:", selected["prepared_non_empty_text_count"])
    print("Candidate count:", len(selection["candidates_ranked"]))

    pd.DataFrame(selection["candidates_ranked"])[[
        "run_root",
        "raw_row_count",
        "hydrated_row_count",
        "unique_uri_count",
        "prepared_non_empty_text_count",
        "text_source_counts",
    ]]

## 2. Prepare Posts and Inspect Schema

One prepared row is created per `uri`, preferring hydrated text when available.


In [ ]:
prepared_df, selection = prepare_from_best_local_source(base_dir=bluesky_source_root)
prepared_df = prepared_df.sort_values(["uri"], kind="stable").reset_index(drop=True)

print("Rows:", len(prepared_df))
print("Columns:", len(prepared_df.columns))
prepared_df.head(5)

In [ ]:
prepared_df.dtypes.to_frame("dtype")


## 3. Preparation Profiling Summary

Includes row counts, blank text counts, duplicate rates, and text-artifact flags.


In [ ]:
summary = summarize_prepared_posts(prepared_df)
raw = prepared_df["post_text_raw"].fillna("").astype(str)
clean = prepared_df["post_text_clean"].fillna("").astype(str)
raw_non_empty = raw.str.strip().ne("")
clean_non_empty = clean.str.strip().ne("")

duplicate_impact = {
    "duplicate_raw_text_rows_non_empty": int(prepared_df.loc[raw_non_empty].duplicated(subset=["post_text_raw"]).sum()),
    "duplicate_clean_text_rows_non_empty": int(prepared_df.loc[clean_non_empty].duplicated(subset=["post_text_clean"]).sum()),
    "unique_non_empty_raw_text_count": int(raw[raw_non_empty].nunique(dropna=True)),
    "unique_non_empty_clean_text_count": int(clean[clean_non_empty].nunique(dropna=True)),
}

{**summary, **duplicate_impact}


## 4. Before/After and Artifact-Focused Examples

Inspect URLs, hashtags, punctuation-heavy rows, multiline text, non-ASCII text, and blank values.


In [ ]:
changed = prepared_df.loc[prepared_df["post_text_raw"] != prepared_df["post_text_clean"], [
    "uri",
    "text_source",
    "post_text_raw",
    "post_text_clean",
    "post_text_alnum",
]].head(12)
changed


In [ ]:
artifact_views = {
    "hashtags": prepared_df.loc[prepared_df["has_hashtag"], ["uri", "post_text_raw", "post_text_clean"]].head(5),
    "urls": prepared_df.loc[prepared_df["has_url"], ["uri", "post_text_raw", "post_text_clean"]].head(5),
    "mentions": prepared_df.loc[prepared_df["has_mention"], ["uri", "post_text_raw", "post_text_clean"]].head(5),
    "non_ascii": prepared_df.loc[prepared_df["has_non_ascii"], ["uri", "post_text_raw", "post_text_clean"]].head(5),
    "blank_clean": prepared_df.loc[prepared_df["post_text_clean"].fillna("").str.strip().eq(""), ["uri", "text_source", "post_text_raw", "post_text_clean"]].head(5),
}
artifact_views


## 5. Save Required Outputs

Write full prepared parquet, sample parquet/csv, and optional compact summary JSON.


In [ ]:
from datetime import datetime, timezone

local_out = ROOT / "local/derived/bluesky"
sample_out = ROOT / "data/samples"
local_out.mkdir(parents=True, exist_ok=True)
sample_out.mkdir(parents=True, exist_ok=True)

full_path = local_out / "bluesky_posts_prepared.parquet"
sample_parquet_path = sample_out / "bluesky_posts_prepared_sample_1000.parquet"
sample_csv_path = sample_out / "bluesky_posts_prepared_sample_1000.csv"
summary_path = local_out / "bluesky_posts_preparation_summary.json"

prepared_df.to_parquet(full_path, index=False)
prepared_df.head(1000).to_parquet(sample_parquet_path, index=False)
prepared_df.head(1000).to_csv(sample_csv_path, index=False)

summary_json = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_source": selection["selected"],
    "selection_rule": selection["selection_rule"],
    "prepared_profile": summarize_prepared_posts(prepared_df),
    "output_paths": {
        "full": str(full_path),
        "sample_parquet": str(sample_parquet_path),
        "sample_csv": str(sample_csv_path),
    },
}
summary_path.write_text(json.dumps(summary_json, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Wrote:", full_path)
print("Wrote:", sample_parquet_path)
print("Wrote:", sample_csv_path)
print("Wrote:", summary_path)

## 6. Readiness Check

If non-empty prepared text coverage is strong and transforms are deterministic, this dataset is ready for Phase 24 local candidate extraction.
